# HW5 Q1: ResNet-20 Quantization-aware Training (CIFAR-10)
4-bit (first conv and FC kept in float) and 2-bit (all convs quantized, FC float).

In [1]:
import os, time
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

from models.quant_layer import act_quantization, QuantConv2d as QuantConv2d4
from models.resnet_quant import resnet20_quant
from models.resnet_quant_2bit import resnet20_quant_2bit
from models.quant_layer_2bit import QuantConv2d as QuantConv2d2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 128
normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms.Compose(
    [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(), transforms.ToTensor(), normalize]))
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor(), normalize]))
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)


def evaluate(model):
    model.eval(); correct = 0
    with torch.no_grad():
        for x, y in testloader:
            correct += model(x.to(device)).argmax(1).eq(y.to(device)).sum().item()
    return 100. * correct / len(testset)


def train_qat(model, name, epochs, lr=0.04, wd=1e-4, retrain=False):
    """QAT with SGD + cosine LR. Skips training if a checkpoint exists. Per-epoch logs are not printed."""
    os.makedirs('result/' + name, exist_ok=True)
    path = 'result/%s/model_best.pth.tar' % name
    if os.path.exists(path) and not retrain:
        model.load_state_dict(torch.load(path)['state_dict']); return model
    model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    crit = nn.CrossEntropyLoss(); best = 0
    for ep in range(epochs):
        model.train()
        for x, y in trainloader:
            x, y = x.to(device), y.to(device)
            loss = crit(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        acc = evaluate(model)
        if acc > best:
            best = acc
            torch.save({'epoch': ep + 1, 'state_dict': model.state_dict(), 'best_prec': best}, path)
    model.load_state_dict(torch.load(path)['state_dict'])
    return model

## (A) 4-bit ResNet-20
`resnet20_quant` uses `QuantConv2d` (bit = 4) in both convs of every BasicBlock and in the two 1x1 downsample convs. `conv1` and `fc` stay float.

In [2]:
model4 = resnet20_quant().to(device)
print(model4.conv1); print(model4.fc); print(model4.layer1[0].conv1)

Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Linear(in_features=64, out_features=10, bias=True)
QuantConv2d(
  16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
  (weight_quant): weight_quantize_fn()
)


In [3]:
model4 = train_qat(model4, 'resnet20_quant_4bit', epochs=80)
print('4-bit test accuracy: %.2f%%' % evaluate(model4))
model4.show_params()

4-bit test accuracy: 90.17%
clipping threshold weight alpha: 2.614000, activation alpha: 4.437000
clipping threshold weight alpha: 2.273000, activation alpha: 4.228000
clipping threshold weight alpha: 2.430000, activation alpha: 4.444000
clipping threshold weight alpha: 2.141000, activation alpha: 4.230000
clipping threshold weight alpha: 2.492000, activation alpha: 4.572000
clipping threshold weight alpha: 2.253000, activation alpha: 4.300000
clipping threshold weight alpha: 2.343000, activation alpha: 4.676000
clipping threshold weight alpha: 2.426000, activation alpha: 4.267000
clipping threshold weight alpha: 2.133000, activation alpha: 4.695000
clipping threshold weight alpha: 2.321000, activation alpha: 4.382000
clipping threshold weight alpha: 2.296000, activation alpha: 4.249000
clipping threshold weight alpha: 2.340000, activation alpha: 4.604000
clipping threshold weight alpha: 2.272000, activation alpha: 4.243000
clipping threshold weight alpha: 2.397000, activation alpha: 4

### x_int and w_int for `layer1[0].conv1` (the 2nd conv layer)
Hook the layer to capture its (float) input and its own output.

In [4]:
model4.eval()
layer = model4.layer1[0].conv1
cap = {}
h = layer.register_forward_hook(lambda m, i, o: cap.update(x=i[0].detach(), out=o.detach()))
data, _ = next(iter(testloader))
with torch.no_grad(): model4(data.to(device))
h.remove()
x, out_layer = cap['x'], cap['out']
print(x.shape, out_layer.shape)

torch.Size([128, 16, 32, 32]) torch.Size([128, 16, 32, 32])


In [5]:
w_bit = 4
weight_q = layer.weight_q.detach()                   # quantized weight stored in forward (normalized units)
w_alpha = layer.weight_quant.wgt_alpha.item()
w_delta = w_alpha / (2 ** (w_bit - 1) - 1)           # signed: 2^(b-1)-1 positive levels
weight_int = torch.round(weight_q / w_delta)
print('w_int range:', weight_int.min().item(), weight_int.max().item(), '| unique:', weight_int.unique().tolist())
print('max |weight_q - w_int*delta|:', (weight_q - weight_int * w_delta).abs().max().item())

w_int range: -7.0 7.0 | unique: [-7.0, -6.0, -5.0, -4.0, -3.0, -2.0, -1.0, -0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
max |weight_q - w_int*delta|: 2.384185791015625e-07


In [6]:
x_bit = 4
x_alpha = layer.act_alpha.item()
x_delta = x_alpha / (2 ** x_bit - 1)                 # unsigned: 2^b-1 levels
x_q = act_quantization(x_bit)(x, layer.act_alpha.detach())
x_int = torch.round(x_q / x_delta)
print('x_int range:', x_int.min().item(), x_int.max().item(), '| unique count:', x_int.unique().numel())

x_int range: 0.0 15.0 | unique count: 16


In [7]:
conv_int = nn.Conv2d(16, 16, kernel_size=3, stride=1, padding=1, bias=False).to(device)
conv_int.weight = nn.Parameter(weight_int)
with torch.no_grad():
    psum_int = conv_int(x_int)
    psum_recovered = psum_int * x_delta * w_delta
print(psum_recovered[0, 0, :3, :3])

diff_layer = (out_layer - psum_recovered).abs()
print('mean |layer output - recovered psum|:', diff_layer.mean().item())
print('max  |layer output - recovered psum|:', diff_layer.max().item())

tensor([[ 5.0813,  2.8720,  0.2209],
        [12.7031,  7.0696,  5.1917],
        [11.1567,  3.0929,  4.4185]], device='cuda:0')
mean |layer output - recovered psum|: 0.004391090478748083
max  |layer output - recovered psum|: 0.04489898681640625


In [8]:
# (reference, as in HW_Code5) recovered psum vs. float input x with quantized weight
conv_ref = nn.Conv2d(16, 16, kernel_size=3, stride=1, padding=1, bias=False).to(device)
conv_ref.weight = nn.Parameter(weight_q)
with torch.no_grad(): out_ref = conv_ref(x)
print('mean |float-x conv - recovered psum|:', (out_ref - psum_recovered).abs().mean().item())

mean |float-x conv - recovered psum|: 0.9844503402709961


## (B) 2-bit ResNet-20
New files: `models/quant_layer_2bit.py` (QuantConv2d with a `bit` argument) and `models/resnet_quant_2bit.py` (all convs 2-bit, including `conv1` whose `act_alpha` starts at 2).

In [9]:
model2 = resnet20_quant_2bit().to(device)
print(model2.conv1)
print(model2.layer1[0].conv1)

QuantConv2d(
  3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
  (weight_quant): weight_quantize_fn()
)
QuantConv2d(
  16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
  (weight_quant): weight_quantize_fn()
)


In [10]:
model2 = train_qat(model2, 'resnet20_quant_2bit', epochs=120)
print('2-bit test accuracy: %.2f%%' % evaluate(model2))
model2.show_params()

2-bit test accuracy: 81.20%
clipping threshold weight alpha: 1.425000, activation alpha: 2.092000
clipping threshold weight alpha: 2.847000, activation alpha: 7.310000
clipping threshold weight alpha: 2.335000, activation alpha: 3.804000
clipping threshold weight alpha: 2.679000, activation alpha: 6.318000
clipping threshold weight alpha: 2.229000, activation alpha: 3.621000
clipping threshold weight alpha: 2.725000, activation alpha: 5.519000
clipping threshold weight alpha: 2.179000, activation alpha: 3.548000
clipping threshold weight alpha: 2.491000, activation alpha: 5.313000
clipping threshold weight alpha: 2.754000, activation alpha: 5.561000
clipping threshold weight alpha: 2.154000, activation alpha: 5.345000
clipping threshold weight alpha: 2.490000, activation alpha: 4.875000
clipping threshold weight alpha: 2.348000, activation alpha: 3.828000
clipping threshold weight alpha: 2.362000, activation alpha: 5.119000
clipping threshold weight alpha: 2.247000, activation alpha: 3

In [11]:
# sanity check of bit widths: 2-bit weights -> -1..1, 2-bit activations -> 0..3
model2.eval(); l2 = model2.layer1[0].conv1; cap = {}
h = l2.register_forward_hook(lambda m, i, o: cap.update(x=i[0].detach()))
with torch.no_grad(): model2(data.to(device))
h.remove()
wi = torch.round(l2.weight_q.detach() / (l2.weight_quant.wgt_alpha.item() / (2 ** 1 - 1)))
xi = torch.round(act_quantization(2)(cap['x'], l2.act_alpha.detach()) / (l2.act_alpha.item() / 3))
print('w_int values:', wi.unique().tolist(), '| x_int values:', xi.unique().tolist())

w_int values: [-1.0, -0.0, 1.0] | x_int values: [0.0, 1.0, 2.0, 3.0]
